# LatticeMemory — MS-MARCO Dual-Encoder E8 Training

**Goal:** Train a full-encoder E8 routing model on MS-MARCO that maps query-passage pairs to nearby E8 cells (asymmetric QA retrieval).

**Config:** `lambda_address=8, lambda_hard=2` — the anti-collapse config. Hard negatives prevent all embeddings collapsing to one cell; address CE trains correct routing.

**Before running this notebook:**
1. Enable GPU runtime: `Runtime → Change runtime type → T4 GPU` (or A100 if on Colab Pro)
2. Push your latest code to GitHub on the `release/hamming-router-productization` branch

**Checkpoints:** Every epoch is saved to Google Drive — if Colab disconnects, reconnect and resume from the last epoch checkpoint.

## 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BASE = '/content/drive/MyDrive/latticememory_training'
os.makedirs(DRIVE_BASE, exist_ok=True)
print(f'Drive mounted. Output will go to: {DRIVE_BASE}')

## 2 — Clone repo and install

In [ ]:
import os

REPO_URL = 'https://github.com/sangmorg1-debug/e8-Project'
BRANCH   = 'release/hamming-router-productization'
REPO_DIR = '/content/latticememory'

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    print('Repo already cloned — pulling latest')
    !git -C {REPO_DIR} pull origin {BRANCH}

os.chdir(REPO_DIR)
!git log --oneline -5

In [ ]:
# Install the package with training extras
!pip install -e ".[training]" -q
# Verify import
import latticememory
print('latticememory imported OK')

## 3 — Check GPU and pick batch size

In [ ]:
import torch

if not torch.cuda.is_available():
    print('WARNING: No GPU detected. Training will be extremely slow on CPU.')
    print('Go to Runtime → Change runtime type → T4 GPU')
    BATCH_SIZE = 4
    GRAD_ACCUM = 4
else:
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name}')
    print(f'VRAM: {vram_gb:.1f} GB')

    # bge-large (1024D, ~330M params) with fp16 + gradient checkpointing:
    #   T4  (16 GB) → batch_size=16 safe
    #   A100(40 GB) → batch_size=32 safe
    #   V100(16 GB) → batch_size=16 safe
    if vram_gb >= 35:
        BATCH_SIZE = 32
        GRAD_ACCUM = 2   # effective batch = 64
    elif vram_gb >= 14:
        BATCH_SIZE = 16
        GRAD_ACCUM = 4   # effective batch = 64
    else:
        BATCH_SIZE = 8
        GRAD_ACCUM = 8   # effective batch = 64

print(f'batch_size={BATCH_SIZE}  grad_accum={GRAD_ACCUM}  effective_batch={BATCH_SIZE * GRAD_ACCUM}')

## 4 — Training configuration

Edit this cell to change hyper-parameters.

In [ ]:
# ── Training hyper-parameters ─────────────────────────────────────────────
RUN_NAME           = 'msmarco_full_encoder_2k_15ep_lam8hard'
MODEL              = 'dfrokido/bge-large-e8-snap'
TRAIN_LIMIT        = 2000     # training Q-A pairs from MS-MARCO
EVAL_LIMIT         = 200      # held-out evaluation pairs
EPOCHS             = 15
LR                 = 2e-5
LAMBDA_ADDRESS     = 8.0      # E8 cell cross-entropy — drives routing alignment
LAMBDA_HARD        = 2.0      # hard negative ranking — prevents key collapse
LAMBDA_NEIGHBORHOOD= 0.5      # expected Hamming — smooth distance regularizer
LOG_EVERY_BATCHES  = 25
# ──────────────────────────────────────────────────────────────────────────

import os
OUTPUT_DIR = f'{DRIVE_BASE}/{RUN_NAME}'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output dir: {OUTPUT_DIR}')
print(f'Checkpoints will be saved every epoch to {OUTPUT_DIR}/checkpoints/epoch_N/')

## 5 — Run training

Output is printed live. Each epoch also logs a JSON line. If Colab disconnects, skip to the **Resume** cell below.

In [ ]:
import subprocess, sys, json

cmd = [
    sys.executable, '-m', 'latticememory.training',
    '--training-mode',              'full_encoder',
    '--model',                      MODEL,
    '--train-limit',                str(TRAIN_LIMIT),
    '--eval-limit',                 str(EVAL_LIMIT),
    '--epochs',                     str(EPOCHS),
    '--batch-size',                 str(BATCH_SIZE),
    '--gradient-accumulation-steps',str(GRAD_ACCUM),
    '--lr',                         str(LR),
    '--lambda-address',             str(LAMBDA_ADDRESS),
    '--lambda-hard',                str(LAMBDA_HARD),
    '--lambda-neighborhood',        str(LAMBDA_NEIGHBORHOOD),
    '--checkpoint-every-epoch',                            # save model every epoch
    '--log-every-batches',          str(LOG_EVERY_BATCHES),
    '--output-dir',                 OUTPUT_DIR,
]

print('Command:', ' '.join(cmd))
print('─' * 70)

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd=REPO_DIR,
)

try:
    for line in proc.stdout:
        print(line, end='', flush=True)
except KeyboardInterrupt:
    proc.terminate()
    print('\nTraining interrupted. Last checkpoint is on Drive.')

proc.wait()
print(f'\nExit code: {proc.returncode}')

## 6 — Resume from last checkpoint

If Colab disconnected, run cells 1–4 first, then this cell. It finds the latest epoch checkpoint on Drive and resumes training from there.

In [ ]:
import os, re, subprocess, sys

CKPT_DIR = f'{OUTPUT_DIR}/checkpoints'

# Find the highest epoch checkpoint
if not os.path.exists(CKPT_DIR):
    print('No checkpoints found. Run the training cell from scratch.')
else:
    epochs_saved = sorted(
        [int(m.group(1)) for d in os.listdir(CKPT_DIR)
         if (m := re.match(r'epoch_(\d+)', d))],
    )
    if not epochs_saved:
        print('Checkpoint dir exists but is empty. Run training cell from scratch.')
    else:
        last_epoch = epochs_saved[-1]
        resume_from = f'{CKPT_DIR}/epoch_{last_epoch}'
        remaining_epochs = EPOCHS - last_epoch
        print(f'Found checkpoints for epochs: {epochs_saved}')
        print(f'Resuming from epoch {last_epoch} — {remaining_epochs} epochs remaining')
        print(f'Resume model path: {resume_from}')

        if remaining_epochs <= 0:
            print('Training already complete!')
        else:
            cmd = [
                sys.executable, '-m', 'latticememory.training',
                '--training-mode',               'full_encoder',
                '--model',                       resume_from,   # load from checkpoint
                '--train-limit',                 str(TRAIN_LIMIT),
                '--eval-limit',                  str(EVAL_LIMIT),
                '--epochs',                      str(remaining_epochs),
                '--batch-size',                  str(BATCH_SIZE),
                '--gradient-accumulation-steps', str(GRAD_ACCUM),
                '--lr',                          str(LR),
                '--lambda-address',              str(LAMBDA_ADDRESS),
                '--lambda-hard',                 str(LAMBDA_HARD),
                '--lambda-neighborhood',         str(LAMBDA_NEIGHBORHOOD),
                '--checkpoint-every-epoch',
                '--log-every-batches',           str(LOG_EVERY_BATCHES),
                '--output-dir',                  f'{OUTPUT_DIR}_resume_ep{last_epoch}',
            ]

            print('\nStarting resumed training...')
            print('─' * 70)

            proc = subprocess.Popen(
                cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                text=True, bufsize=1, cwd=REPO_DIR,
            )
            try:
                for line in proc.stdout:
                    print(line, end='', flush=True)
            except KeyboardInterrupt:
                proc.terminate()
                print('\nInterrupted. Last checkpoint saved to Drive.')
            proc.wait()
            print(f'\nExit code: {proc.returncode}')

## 7 — Results: training curves and final metrics

In [ ]:
import json, os
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

metrics_path = f'{OUTPUT_DIR}/metrics.json'

if not os.path.exists(metrics_path):
    print(f'metrics.json not found yet at {metrics_path}')
    print('Training may still be running, or it ended early — check the output above.')
else:
    m = json.load(open(metrics_path))

    print('═' * 60)
    print('FINAL METRICS')
    print('═' * 60)
    for split in ('train', 'eval'):
        s = m.get(split, {})
        if not s:
            continue
        print(f'\n{split.upper()} ({s.get("total","?")} examples):')
        print(f'  recall@1              = {s.get("recall_at_1","?")}')
        print(f'  mean Hamming distance = {s.get("mean_hamming_distance","?")}')
        print(f'  route rate            = {s.get("lattice_route_rate","?")}')
        print(f'  unique document keys  = {s.get("unique_document_keys","?")}')
        print(f'  max docs per key      = {s.get("max_documents_per_key","?")}')
        print(f'  collision key count   = {s.get("collision_key_count","?")}')

    # Training curves
    hamming_hist = m.get('train_mean_hamming_history', [])
    loss_hist    = m.get('train_loss_history', [])

    if hamming_hist or loss_hist:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        epochs = list(range(1, len(hamming_hist) + 1))

        if hamming_hist:
            axes[0].plot(epochs, hamming_hist, 'o-', color='steelblue')
            axes[0].axhline(0, color='red', linestyle='--', alpha=0.4, label='collapse threshold')
            axes[0].set_xlabel('Epoch')
            axes[0].set_ylabel('Mean Hamming (train)')
            axes[0].set_title('Hamming Distance — Training Set')
            axes[0].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
            axes[0].legend()

        if loss_hist:
            axes[1].plot(list(range(1, len(loss_hist) + 1)), loss_hist, 'o-', color='darkorange')
            axes[1].set_xlabel('Epoch')
            axes[1].set_ylabel('Total Loss')
            axes[1].set_title('Training Loss')
            axes[1].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

        plt.tight_layout()
        plt.savefig(f'{OUTPUT_DIR}/training_curves.png', dpi=120)
        plt.show()
        print(f'Saved: {OUTPUT_DIR}/training_curves.png')

## 8 — Live epoch monitor (while training runs)

Run this cell in a **separate** browser tab while training is running to watch per-epoch progress without interrupting the training cell.

In [ ]:
import json, os, time

stdout_log = f'{OUTPUT_DIR}/stdout.log'   # only exists if you redirected — usually not needed
metrics_f  = f'{OUTPUT_DIR}/metrics.json'

# Watch the stdout.log line-by-line for checkpoint events
# (Only works if training was launched with stdout redirect — otherwise read from cell output)
ckpt_dir = f'{OUTPUT_DIR}/checkpoints'

print('Watching for epoch checkpoints on Drive...')
print('(Re-run this cell to refresh)')
print()

if os.path.exists(ckpt_dir):
    import re
    epochs_done = sorted(
        [int(m.group(1)) for d in os.listdir(ckpt_dir)
         if (m := re.match(r'epoch_(\d+)', d))]
    )
    print(f'Epochs saved to Drive: {epochs_done}')
    if epochs_done:
        last = epochs_done[-1]
        ckpt_contents = os.listdir(f'{ckpt_dir}/epoch_{last}')
        print(f'Latest checkpoint (epoch {last}): {ckpt_contents}')
else:
    print('No checkpoints yet.')

if os.path.exists(metrics_f):
    d = json.load(open(metrics_f))
    hh = d.get('train_mean_hamming_history', [])
    lh = d.get('train_loss_history', [])
    print(f'\nHamming history:  {hh}')
    print(f'Loss history:     {lh}')
else:
    print('\nmetrics.json not yet written (training still running).')

## 9 — What to look for

| Signal | Meaning |
|--------|----------|
| `mean_hamming` declining steadily but stays > 0 for many epochs | **Good** — model converging without collapse |
| `mean_hamming` → 0 and `unique_document_keys` is tiny (1–5) | **Collapse** — stop, reduce `LAMBDA_ADDRESS` |
| `mean_hamming` stays near 100 after epoch 5+ | **Not learning** — try reducing `LAMBDA_HARD` |
| `recall@1` > 0.30 with `unique_document_keys` close to `EVAL_LIMIT` | **Real routing signal** — scale up |
| `recall@1` > 0.10 and `unique_document_keys` healthy | **Promising** — training is working |

### Baseline context
- Prior run (100 examples, 5 epochs, same lambdas): `recall@1=0.06`, `hamming=4.06` — under-trained, still converging
- DPR (Dense Passage Retrieval) recall@10 on MS-MARCO: ~78% (our goal is to match or approach this with E8 keys)
- Random baseline (128 blocks): Hamming ≈ 88–106 for any unrelated pair